## 参考文档

1. 收盘前逆回股
https://www.xuntou.net/forum.php?mod=viewthread&tid=192&extra=&user_code=3GficF

2. 常见问题
https://www.xuntou.net/forum.php?mod=viewthread&tid=66&extra=page%3D1&user_code=3GficF

3. 如何删除本地数据

https://www.xuntou.net/forum.php?mod=viewthread&tid=185&extra=page%3D1&user_code=3GficF

4. qmt 封装

https://mp.weixin.qq.com/s/EFEBtmT9uy8MLG-Nca_apw

5. 复权计算
https://www.xuntou.net/forum.php?mod=viewthread&tid=198&extra=&user_code=3GficF

## 获取数据

In [21]:
from xtquant import xtdata as xt
xt.download_etf_info()


RuntimeError: 褰撳墠瀹㈡埛绔湭鏀寔姝ゅ姛鑳斤紝璇锋洿鏂板鎴风鎴栧崌绾ф姇鐮旂増

In [5]:
from xtquant import xtdata
stocks = ['000001.SZ', '600000.SH']

xtdata.download_history_data(stocks[0], '1d')
xtdata.download_history_data(stocks[1], '1d')

end = "20050104"
bars = xtdata.get_market_data(stock_list=stocks, period='1d', start_time=end, count=-1, dividend_type="front_ratio")

display(bars['close'].T.tail())


,000001.SZ,600000.SH
20240304,10.33,7.07
20240305,10.43,7.16
20240306,10.33,7.12
20240307,10.38,7.14
20240308,10.38,7.12


In [22]:
import datetime

now = datetime.datetime.now()
print(now.strftime("%Y%m%d%H%M%S"))

import arrow
arrow.get(now).format("YYYYMMDDHHmmss")


20240308220549


'20240308220549'

In [ ]:
from xtquant.xtdata import *

stocks = ['000001.SZ', '600000.SH']

download_history_data(stocks[0], 'tick', start_time="20230604", incrementally=True)
download_history_data(stocks[1], 'tick',start_time="20230604", incrementally=True)



In [ ]:
end = "20231206"
bars = get_market_data(stock_list=stocks, period='tick', end_time=end)


In [ ]:
bars['000001.SZ']


In [ ]:
get_market_data?


In [ ]:
from xtquant.xtdata import download_history_data2

def on_progress(data):
    '''补充历史数据回调函数'''
    print(data) 
    

stock_list = ['603909.SH','300450.SZ','600740.SH']
field_list = ['time','open','close','low','high','volume']

# download_history_data2(stock_list, period='1d', start_time='20230201', end_time='20230223', callback=on_progress)

print('download_history_data2 finished')

# 获取股票close数据
ret = get_market_data(field_list, stock_list, period='1d', start_time='', end_time='', count=5, dividend_type='front', fill_data=True)
print(ret['close'].T)


## 除权数据

In [ ]:
code = "000001.SZ"
get_divid_factors(code, start_time='20050104', end_time='')


In [26]:
#coding:utf-8

import numpy as np
import pandas as pd

from xtquant import xtdata


def gen_divid_ratio(quote_datas, divid_datas):
    drl = []
    dr = 1.0
    qi = 0
    qdl = len(quote_datas)
    di = 0
    ddl = len(divid_datas)
    while qi < qdl and di < ddl:
        qd = quote_datas.iloc[qi]
        dd = divid_datas.iloc[di]
        if qd.name >= dd.name:
            dr *= dd['dr']
            di += 1
        if qd.name <= dd.name:
            drl.append(dr)
            qi += 1
    while qi < qdl:
        drl.append(dr)
        qi += 1
    return pd.DataFrame(drl, index = quote_datas.index, columns = quote_datas.columns)

def process_forward_ratio(quote_datas, divid_datas):
    drl = gen_divid_ratio(quote_datas, divid_datas)
    drlf = drl / drl.iloc[-1]
    result = (quote_datas * drlf).apply(lambda x: round(x, 2))
    return result

def process_backward_ratio(quote_datas, divid_datas):
    drl = gen_divid_ratio(quote_datas, divid_datas)
    result = (quote_datas * drl).apply(lambda x: round(x, 2))
    return result

def process_forward(quote_datas1, divid_datas):
    quote_datas = quote_datas1.copy()
    def calc_front(v, d):
        return round(
            (v - d['interest'] + d['allotPrice'] * d['allotNum'])
            / (1 + d['allotNum'] + d['stockBonus'] + d['stockGift'])
            , 2
        )
    for qi in range(len(quote_datas)):
        q = quote_datas.iloc[qi]
        for di in range(len(divid_datas)):
            d = divid_datas.iloc[di]
            if d.name <= q.name:
                continue
            q.iloc[0] = calc_front(q.iloc[0], d)
    return quote_datas

def process_backward(quote_datas1, divid_datas):
    quote_datas = quote_datas1.copy()
    def calc_front(v, d):
        return round(
            (v * (1 + d['stockGift'] + d['stockBonus'] + d['allotNum'])
            + d['interest'] - d['allotNum'] * d['allotPrice'])
            , 2
        )
    for qi in range(len(quote_datas)):
        q = quote_datas.iloc[qi]
        for di in range(len(divid_datas)):
            d = divid_datas.iloc[di]
            if d.name > q.name:
                continue
            q.iloc[0] = calc_front(q.iloc[0], d)
    return quote_datas


#--------------------------------

s = '002594.SZ'

xtdata.download_history_data(s, '1d', '20220708', '20240308')

dd = xtdata.get_divid_factors(s)
print(dd)

#复权计算用于处理价格字段
field_list = ['open', 'high', 'low', 'close']
datas_ori = xtdata.get_market_data(field_list, [s], '1d', dividend_type = 'none', start_time='20220708', end_time='20240308')['close'].T
#print(datas_ori)


# #等比前复权
# datas_forward_ratio = process_forward_ratio(datas_ori, dd)
# print('datas_forward_ratio', datas_forward_ratio)

# #等比后复权
# datas_backward_ratio = process_backward_ratio(datas_ori, dd)
# print('datas_backward_ratio', datas_backward_ratio)

# #前复权
# datas_forward = process_forward(datas_ori, dd)
# print('datas_forward', datas_forward)

# #后复权
# datas_backward = process_backward(datas_ori, dd)
# print('datas_backward', datas_backward)

factors = gen_divid_ratio(datas_ori, dd)
factors



                  time  interest  stockBonus  stockGift  allotNum  allotPrice  \
20140813  1.407859e+12     0.050         0.0        0.0       0.0         0.0   
20161213  1.481558e+12     0.367         0.0        0.0       0.0         0.0   
20170804  1.501776e+12     0.178         0.0        0.0       0.0         0.0   
20180817  1.534435e+12     0.141         0.0        0.0       0.0         0.0   
20190726  1.564070e+12     0.204         0.0        0.0       0.0         0.0   
20200818  1.597680e+12     0.060         0.0        0.0       0.0         0.0   
20210806  1.628179e+12     0.148         0.0        0.0       0.0         0.0   
20220729  1.659024e+12     0.105         0.0        0.0       0.0         0.0   
20230728  1.690474e+12     1.142         0.0        0.0       0.0         0.0   

          gugai        dr  
20140813    0.0  1.000970  
20161213    0.0  1.007482  
20170804    0.0  1.003712  
20180817    0.0  1.003323  
20190726    0.0  1.003533  
20200818    0.0  1.00

,002594.SZ
20220708,1.020403
20220711,1.020403
20220712,1.020403
20220713,1.020403
20220714,1.020403
...,...
20240304,1.025215
20240305,1.025215
20240306,1.025215
20240307,1.025215


In [25]:
len(datas_ori)

672

In [15]:
import datetime
def calc_factors(code, end: datetime.date):
    pass

# diff_factors = xtdata.get_divid_factors("000001.SZ")
diff_factors.index[0]



'19910403'

In [19]:
# init cache

from pyqmt.dal import init_dal
import os
import cfg4py
os.environ[cfg4py.envar] = "DEV"
init_dal()

cfg = cfg4py.get_instance()

2024-03-08 21:19:55,935 I cfg4py.core:update_config:280 | configuration is
chores_db_path: d:\data\pyqmt.db
clickhouse: {database: zillionare, host: 192.168.100.10, password: 123456, user: default}
redis: {host: 192.168.100.10, port: 6379}
server: {prefix: pyqmt}



In [20]:
now = datetime.datetime.now()


'2024'

## 如果数据没有缓存，则get_market_data会失败

In [ ]:
ret = get_market_data(field_list, ['000004.SZ'], period='1d', start_time='', count=5, dividend_type='front', fill_data=True)

print(ret['close'].T)
print(ret)


In [ ]:
import pandas as pd
import datetime
from xtquant import xtdata

def get_tick(code, start_time, end_time, period='tick'):
    xtdata.download_history_data(code, period=period, start_time=start_time, end_time=end_time)
    data = xtdata.get_local_data(field_list=[], stock_list=[code], period=period, count=10)
    result_list = data[code]
    df = pd.DataFrame(result_list)

    df['time_str'] = df['time'].apply(lambda x: datetime.datetime.fromtimestamp(x / 1000.0))
    return df


def process_timestamp(df, filename):
    df = df.set_index('time_str')
    result = df.resample('3S').first().ffill()
    # result = result[(result.index >= '2022-07-20 09:30') & (result.index <= '2022-07-20 15:00')]
    result = result.reset_index()
    result.to_csv(filename + '.csv')


def dump_single_code_tick():
    # 导出单个转债的tick数据
    code='128022'
    start_date = '20210113'
    end_date = '20210130'

    post_fix = 'SZ' if code.startswith('12') else 'SH'
    code = '{}.{}'.format(code,post_fix)
    df = get_tick(code, start_date, end_date)
    return df

dump_single_code_tick()


## get_full_tick

In [ ]:
from xtquant.xtdata import get_full_tick,get_divid_factors

get_full_tick(['SH'])
get_divid_factors('000001.SZ', start_time='', end_time='')


In [ ]:
from xtquant.xtdata import get_l2_quote
get_l2_quote(field_list=[], stock_code='', start_time='', end_time='', count=-1)


In [ ]:
code = '000001.SZ'
get_instrument_detail(code)


## 获取交易日历
注意返回时间是毫秒单位，如果我们用datetime.datetime来转换，需要除以1000
也可以用numpy来转换

In [ ]:
days = get_trading_dates('SH', start_time='', end_time='', count=10)
import numpy as np
np.array(days, dtype='datetime64[ms]').astype(datetime.date)



In [ ]:
get_trading_dates('SH', end_time='20230908', count=10)


### 时间戳转换函数

In [ ]:

import time
def conv_time(ct):
    '''
    conv_time(1476374400000) --> '20161014000000.000'
    '''
    local_time = time.localtime(ct / 1000)
    data_head = time.strftime('%Y%m%d%H%M%S', local_time)
    data_secs = (ct - int(ct)) * 1000
    time_stamp = '%s.%03d' % (data_head, data_secs)
    return time_stamp

conv_time(1693152000000)


In [ ]:
import time
time.time()

import datetime

datetime.datetime.fromtimestamp(1694448000)


In [ ]:
from xtquant import xtdata
sectors = xtdata.get_sector_list()
print(len(sectors))
for i in range(0, len(sectors), 6):
    print(" ".join(sectors[i:i+6]))


## 获取板块列表
析块列表没有索引，以字符串格式返回，也没有时间标签
获取板块名字后，直接通过板块名字来获取成份股

In [ ]:
from xtquant import xtdata
for sector in xtdata.get_sector_list():
    if sector.find("可转债") != -1:
        print(sector)
xtdata.get_stock_list_in_sector("沪深A股")
xtdata.get_stock_list_in_sector("可转债等权")
xtdata.get_instrument_detail("110047.SH")


In [ ]:
index = []
for sector in xtdata.get_sector_list():
    if "迅投" in sector:
        print(sector)

for code in xtdata.get_stock_list_in_sector("迅投一级行业板块指数"):
    detail = xtdata.get_instrument_detail(code)
    name = detail["InstrumentName"]
    if name.startswith("SW"):
        print(code, name)
        


# secs = xtdata.get_stock_list_in_sector('GNoled材料')
# xtdata.download_history_data2(secs, period='1d', start_time="20231220")

# barss = xtdata.get_market_data(stock_list=secs, count=10)
# barss["close"]


In [ ]:
xtdata.get_instrument_detail("SW1电力设备")


In [ ]:
from collections import defaultdict

sectors = defaultdict(list)
prefixes = [
    '1000SW', '500SW', '300SW', 'CSRC', 
    '迅投一级', '迅投二级', '迅投三级', 
    'HKSW', 'TGN', 'TDY', 'THY', 'DY1', 
    'DY2', 'SW1', 'SW2', 'SW3', 'ETF',
    'GN', '转债', '国证', '上证', '沪深', 
    '中证','深证'
]

download_sector_data()

for item in get_sector_list():
    for i in range(6, 1, -1):
        key = item[:i]

        if key in prefixes:
            sectors[key].append(item)
            break
    else:
        sectors['NA'].append(item)

for key, codes in sectors.items():
    print(key, len(codes))


### 可转债

In [ ]:
for sector in xtdata.get_sector_list():
    if sector.find("可转债") != -1:
        print(sector)

convertibles = xtdata.get_stock_list_in_sector("可转债等权")
instrument = xtdata.get_instrument_detail(convertibles[0])
exg = instrument["ExchangeID"]
_id = instrument["InstrumentID"]
name = instrument["InstrumentName"]

print(_id + exg, name)


In [ ]:
print(len(xtdata.get_stock_list_in_sector('沪深指数')))
for sector in xtdata.get_stock_list_in_sector('沪深指数'):
    detail = xtdata.get_instrument_detail(sector)
    name = detail["InstrumentName"]
    if name == "中证1000":
        print(sector)

xtdata.download_history_data("399852.SZ", period="1d")
xtdata.get_market_data(stock_list=["000050.SH"], period='1d', count=10)



In [ ]:
xtdata.get_market_data(stock_list=[sector])


## 获取指数列表及指数行情

In [ ]:
xtdata.get_stock_list_in_sector("沪深指数")
xtdata.get_instrument_type("000100.SH")
xtdata.get_instrument_detail("000100.SH")

xtdata.download_history_data("000100.SH", '1d')
xtdata.get_market_data(stock_list=["000100.SH"], period="1d")


## 获取历史涨跌停数据

In [ ]:
from xtquant import xtdata
xtdata.download_history_data(stock_code="600000.SH", period="stoppricedata")


In [ ]:
download_sector_data()


In [ ]:
len(get_stock_list_in_sector('沪深A股'))


In [ ]:
from collections import defaultdict
from xtquant.xtdata import *

sectors = defaultdict(list)
prefixes = [
    '1000SW', '500SW', '300SW', 'CSRC', '迅投一级', '迅投二级', '迅投三级', 'HKSW',
'TGN', 'TDY', 'THY', 'DY1', 'DY2', 'SW1', 'SW2', 'SW3', 'ETF','GN', '转债', '国证', '上证', '沪深', '中证','深证'
]

sectors = set()

for item in get_sector_list():
    for i in range(6, 1, -1):
        key = item[:i]

        if key.startswith("SW1"):
            sectors.add(key)
            break


print(sectors)
# all_stocks = get_stock_list_in_sector('沪深A股')
# print(f"当前共计{len(all_stocks)}股A股")
# print(all_stocks[:5])


In [ ]:
xt.get_stock_list_in_sector("SW1煤炭")

# coding=utf-8
from xtquant import xtdata
# 下载权重相关信息
xtdata.download_index_weight()
# 获取权重相关信息
xtdata.get_index_weight('SW1煤炭')


In [ ]:
from xtquant.xtdata import get_instrument_detail
get_instrument_detail('000001.SZ')


In [ ]:
#coding=utf-8
from xtquant.xttrader import XtQuantTrader, XtQuantTraderCallback
from xtquant.xttype import StockAccount
from xtquant import xtconstant
import numpy as np


class MyXtQuantTraderCallback(XtQuantTraderCallback):
    def on_disconnected(self):
        """
        连接断开
        :return:
        """
        print("connection lost")

    def on_stock_order(self, order):
        """
        委托回报推送
        :param order: XtOrder对象
        :return:
        """
        print("on order callback:")
        print(order.stock_code, order.order_status, order.order_sysid)
 
    def on_stock_trade(self, trade):
        """
        成交变动推送
        :param trade: XtTrade对象
        :return:
        """
        print("on trade callback")
        print(trade.account_id, trade.stock_code, trade.order_id)

    def on_stock_position(self, position):
        """
        持仓变动推送  注意，该回调函数目前不生效
        :param position: XtPosition对象
        :return:
        """
        print("on position callback")
        print(position.stock_code, position.volume)

    def on_order_error(self, order_error):
        """
        委托失败推送
        :param order_error:XtOrderError 对象
        :return:
        """
        print("on order_error callback")
        print(order_error.order_id, order_error.error_id, order_error.error_msg)

    def on_cancel_error(self, cancel_error):
        """
        撤单失败推送
        :param cancel_error: XtCancelError 对象
        :return:
        """
        print("on cancel_error callback")
        print(cancel_error.order_id, cancel_error.error_id, cancel_error.error_msg)

    def on_order_stock_async_response(self, response):
        """
        异步下单回报推送
        :param response: XtOrderResponse 对象
        :return:
        """
        print("on_order_stock_async_response")
        print(response.account_id, response.order_id, response.seq)

    def on_account_status(self, status):
        """
        :param response: XtAccountStatus 对象
        :return:
        """
        print("on_account_status")
        print(status.account_id, status.account_type, status.status)

# path为mini qmt客户端安装目录下userdata_mini路径
path = 'D:\\国金QMT交易端模拟\\userdata_mini'
session_id = np.random.randint(20000, 60000)
xt_trader = XtQuantTrader(path, session_id)

# path为mini qmt客户端安装目录下userdata_mini路径
path = 'D:\\国金QMT交易端模拟\\userdata_mini'

session_id = np.random.randint(20000, 60000)
xt_trader = XtQuantTrader(path, session_id)

# 创建交易回调类对象，并声明接收回调
callback = MyXtQuantTraderCallback()
xt_trader.register_callback(callback)

# 启动交易线程
xt_trader.start()
# 建立交易连接，返回0表示连接成功
connect_result = xt_trader.connect()
assert connect_result == 0

acc = StockAccount('55010149', 'STOCK')
# 对交易回调进行订阅，订阅后可以收到交易主推，返回0表示订阅成功
subscribe_result = xt_trader.subscribe(acc)
assert subscribe_result == 0


In [ ]:

stock_code = '600000.SH'
# 使用指定价下单，接口返回订单编号，后续可以用于撤单操作以及查询委托状态
print("order using the fix price:")
fix_result_order_id = xt_trader.order_stock(acc, stock_code, xtconstant.STOCK_BUY, 200, xtconstant.FIX_PRICE, 10.5, 'strategy_name', 'remark')
print(fix_result_order_id)

# 使用订单编号撤单
print("cancel order:")
cancel_order_result = xt_trader.cancel_order_stock(acc, fix_result_order_id)
print(cancel_order_result)

# 使用异步下单接口，接口返回下单请求序号seq，seq可以和on_order_stock_async_response的委托反馈response对应起来
print("order using async api:")
async_seq = xt_trader.order_stock_async(acc, stock_code, xtconstant.STOCK_BUY, 200, xtconstant.FIX_PRICE, 10.5, 'strategy_name', 'remark')
print(async_seq)

# 查询证券资产
print("query asset:")
asset = xt_trader.query_stock_asset(acc)
if asset:
    print("asset:")
    print("cash {0}".format(asset.cash))

# 根据订单编号查询委托
print("query order:")
order = xt_trader.query_stock_order(acc, fix_result_order_id)
if order:
    print("order:")
    print("order {0}".format(order.order_id))

    # 阻塞线程，接收交易推送
    # xt_trader.run_forever()


In [ ]:
xt_trader.order_stock?


In [ ]:
import akshare

akshare.stock_hot_rank_detail_realtime_em("SZ000665")
